# OSSI Surface Elevation — High-Pass Filter

Analyze detrended OSSI surface elevation data and apply a high-pass filter to isolate ship-wake-band waves (period < 3.0 s).

- Source: `data/compare_JI/OSSI/h_detrend.csv` (already detrended to remove tide variation)
- Sampling frequency: 5 Hz
- Window: 5 minutes **centered on** `2023-03-10 08:55:45`
- High-pass cutoff: `1 / 3.0 Hz ≈ 0.333 Hz` (keep T < 3 s)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, welch

## 1. Load data and slice 5-minute window

In [ ]:
FS = 5.0  # Hz
T_CUTOFF = 3.0  # s — exclude periods longer than this
FC = 1.0 / T_CUTOFF  # high-pass cutoff frequency (Hz)

WINDOW_DURATION = pd.Timedelta(minutes=5)
WINDOW_CENTER = pd.Timestamp("2023-03-10 08:55:45")
WINDOW_START = WINDOW_CENTER - WINDOW_DURATION / 2
WINDOW_END = WINDOW_CENTER + WINDOW_DURATION / 2

csv_path = "data/compare_JI/OSSI/h_detrend.csv"
df = pd.read_csv(csv_path, parse_dates=["Time"])
df = df.set_index("Time").sort_index()
print(f"Full record: {df.index.min()} → {df.index.max()}  ({len(df):,} samples)")

In [ ]:
win = df.loc[WINDOW_START:WINDOW_END - pd.Timedelta(seconds=1/FS)]
eta = win["SurfaceElevation"].to_numpy()
t = (win.index - WINDOW_START).total_seconds().to_numpy()

expected = int(WINDOW_DURATION.total_seconds() * FS)
print(f"Window samples: {len(eta)}  (expected {expected})")
print(f"η  mean={eta.mean():+.4f} m   std={eta.std():.4f} m   range=[{eta.min():+.3f}, {eta.max():+.3f}] m")

## 2. Design and apply the high-pass filter

4th-order Butterworth, applied zero-phase via `sosfiltfilt` so the filtered signal stays time-aligned with the raw record.

In [ ]:
ORDER = 4
sos = butter(ORDER, FC, btype="highpass", fs=FS, output="sos")
eta_hp = sosfiltfilt(sos, eta)

print(f"High-pass: order={ORDER}, fc={FC:.3f} Hz (T={T_CUTOFF:.1f} s)")
print(f"Filtered η: std={eta_hp.std():.4f} m   range=[{eta_hp.min():+.3f}, {eta_hp.max():+.3f}] m")

## 3. Time-series comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(t, eta, lw=0.7, color="steelblue", alpha=0.7, label="raw")
ax.plot(t, eta_hp, lw=0.8, color="crimson", label=f"high-pass (T < {T_CUTOFF:.1f} s)")

ax.set_xlabel(f"Time (s) from window start ({WINDOW_START})")
ax.set_ylabel("η (m)")
ax.set_title(f"OSSI surface elevation — 5 min centered on {WINDOW_CENTER}")
ax.axhline(0, color="k", lw=0.5)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")

fig.tight_layout()
plt.show()

## 4. Power spectral density (verify cutoff)

In [ ]:
f_raw, P_raw = welch(eta, fs=FS, nperseg=min(512, len(eta)))
f_hp, P_hp = welch(eta_hp, fs=FS, nperseg=min(512, len(eta_hp)))

# Convert to period (drop the f=0 bin so 1/f is finite)
T_raw = 1.0 / f_raw[1:]
T_hp = 1.0 / f_hp[1:]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.loglog(T_raw, P_raw[1:], label="raw", color="steelblue")
ax.loglog(T_hp, P_hp[1:], label=f"high-pass (T < {T_CUTOFF:.1f} s)", color="crimson")
ax.axvline(T_CUTOFF, color="k", ls="--", lw=1, label=f"cutoff = {T_CUTOFF:.1f} s")
ax.set_xlabel("Period T (s)")
ax.set_ylabel("PSD (m²/Hz)")
ax.set_title("Welch PSD — raw vs high-pass")
ax.invert_xaxis()  # long periods on the left → short on the right (low → high frequency)
ax.legend()
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
plt.show()